# LeetCode 4037 — Maximum Valid Split Positions II bằng Python

- Link: https://leetcode.com/problems/maximum-valid-split-positions-ii/
- Độ khó: **Hard**
- Chủ đề: Toán (GCD), Tiền tố / hậu tố (prefix / suffix), Liệt kê ứng viên (enumeration), Lý thuyết số

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxValidSplits(self, nums: list[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums`. Bạn được **xoá tối đa một phần tử**. Với mảng `arr` còn lại, một **vị trí cắt** `i` là hợp lệ nếu `gcd(arr[0..i]) == gcd(arr[i+1..m-1])` (ước chung lớn nhất của phần trái bằng của phần phải). **Điểm** của `arr` là số vị trí cắt hợp lệ; ta cần điểm lớn nhất có thể.

Bài này có hai "tầng": tầng 1 là **tính điểm của một mảng cho trước** (prefix/suffix GCD, và một nhận xét đẹp: điểm có công thức đóng); tầng 2 là **không thể thử xoá từng phần tử** (`n` tới `10^5`) nên phải chứng minh chỉ cần thử `O(log M)` phần tử "đáng nghi".

Sau notebook này bạn sẽ:

1. Dùng `math.gcd`, hiểu `gcd(0, x) = x` và cách tính GCD tiền tố / hậu tố.
2. Chứng minh: vị trí cắt hợp lệ ⇔ cả hai phía có GCD bằng GCD **toàn mảng**, từ đó điểm = `b - a`.
3. Viết lời giải vét cạn `O(n²)` làm "trọng tài".
4. Hiểu vì sao dãy GCD tiền tố chỉ đổi giá trị `O(log M)` lần, và vì sao chỉ cần thử xoá những chỉ số mà GCD tiền tố / hậu tố **thay đổi**.
5. Phân tích độ phức tạp `O(n log² M)` và tự kiểm chứng bằng test ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Bạn được xoá **nhiều nhất một** phần tử của `nums` (có thể không xoá). Các phần tử còn lại giữ nguyên thứ tự, tạo thành `arr` có độ dài `m`.
- Vị trí cắt `i` của `arr` (với `0 <= i < m - 1`) là **hợp lệ** nếu `gcd(arr[0..i]) == gcd(arr[i+1..m-1])`.
- Mảng độ dài 1 không có vị trí cắt nào. **Điểm** của `arr` = số vị trí cắt hợp lệ.
- Trả về **điểm lớn nhất** có thể đạt.

**Ví dụ 1:** `nums = [10, 30, 15, 10]` → `2`. Xoá `15` được `[10, 30, 10]`: cắt tại 0: `gcd(10) = 10`, `gcd(30, 10) = 10`; cắt tại 1: `gcd(10, 30) = 10`, `gcd(10) = 10`. Cả hai hợp lệ.

**Ví dụ 2:** `nums = [2, 10, 14]` → `1`. Không xoá gì: cắt tại 0: `2` vs `gcd(10, 14) = 2` ✔; cắt tại 1: `gcd(2, 10) = 2` vs `14` ✘.

**Ví dụ 3:** `nums = [2, 4]` → `0`.

**Ràng buộc**

- `2 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`

In [1]:
from math import gcd
from functools import reduce
from typing import List

EXAMPLES = [
    ([10, 30, 15, 10], 2),
    ([2, 10, 14], 1),
    ([2, 4], 0),
]
for nums, expected in EXAMPLES:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[10, 30, 15, 10] -> kỳ vọng 2
nums=[2, 10, 14] -> kỳ vọng 1
nums=[2, 4] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `math.gcd` và phần tử "trung tính" `0`

`gcd(a, b)` là ước chung lớn nhất. Điều quan trọng: `gcd(0, x) = x`. Vì vậy ta có thể bắt đầu việc tính GCD của cả dãy bằng `g = 0` rồi lần lượt `g = gcd(g, x)` — không cần xử lý phần tử đầu riêng.

In [2]:
print(gcd(12, 18), gcd(7, 5), gcd(0, 9), gcd(9, 0))

g = 0
for x in [12, 18, 30]:
    g = gcd(g, x)
    print("sau khi thêm", x, "-> gcd =", g)

6 1 9 9
sau khi thêm 12 -> gcd = 12
sau khi thêm 18 -> gcd = 6
sau khi thêm 30 -> gcd = 6


### 3.2. GCD của cả một dãy: `reduce` và vòng lặp

`reduce(gcd, list)` gộp lần lượt từ trái sang phải. Với dãy rỗng cần giá trị khởi tạo `0`: `reduce(gcd, [], 0) = 0`.

In [3]:
print(reduce(gcd, [12, 18, 30]))
print(reduce(gcd, [], 0))
a = [8, 12, 20]
print(reduce(gcd, a[:2]), reduce(gcd, a[1:]))   # gcd của một đoạn con = reduce trên lát cắt

6
0
4 4


### 3.3. Mảng GCD tiền tố `pre` và hậu tố `suf`

- `pre[i] = gcd(arr[0..i])`
- `suf[i] = gcd(arr[i..m-1])`

Khi đó vị trí cắt `i` hợp lệ ⇔ `pre[i] == suf[i + 1]`. Cả hai mảng tính trong `O(m)` lần gọi `gcd`.

In [4]:
def prefix_suffix(arr):
    m = len(arr)
    pre, suf = [0] * m, [0] * m
    g = 0
    for i in range(m):
        g = gcd(g, arr[i]); pre[i] = g
    g = 0
    for i in range(m - 1, -1, -1):
        g = gcd(g, arr[i]); suf[i] = g
    return pre, suf

arr = [10, 30, 10]
pre, suf = prefix_suffix(arr)
print("pre =", pre)
print("suf =", suf)
print("cắt hợp lệ tại:", [i for i in range(len(arr) - 1) if pre[i] == suf[i + 1]])

pre = [10, 10, 10]
suf = [10, 10, 10]
cắt hợp lệ tại: [0, 1]


### 3.4. GCD tiền tố chỉ đổi giá trị rất ít lần

`pre[i] = gcd(pre[i-1], arr[i])` nên `pre[i]` **chia hết** `pre[i-1]`. Nếu `pre[i] != pre[i-1]` thì `pre[i] <= pre[i-1] / 2` (một ước thật sự của số nhỏ hơn hoặc bằng nửa). Do đó từ giá trị ban đầu `<= 10^9` dãy đổi giá trị **không quá `log2(10^9) ≈ 30`** lần.

In [5]:
import random
rng = random.Random(1)
nums = [rng.randint(1, 10**9) for _ in range(10)]
pre, _ = prefix_suffix(nums)
print(pre)                                    # nhanh chóng rơi về 1
chain = [2**29, 2**28, 2**27, 2**20, 2**5, 1]
pre, _ = prefix_suffix(chain)
print(pre, "-> số lần đổi:", sum(pre[i] != pre[i - 1] for i in range(1, len(pre))))

[144272510, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[536870912, 268435456, 134217728, 1048576, 32, 1] -> số lần đổi: 5


### 3.5. Xoá một phần tử khỏi list: cắt lát và nối

`nums[:k] + nums[k + 1:]` là list mới **không có** phần tử ở chỉ số `k` (và `nums` gốc không bị đổi). Đây là cách đơn giản để "thử xoá phần tử k".

In [6]:
nums = [10, 30, 15, 10]
k = 2
print(nums[:k], nums[k + 1:], nums[:k] + nums[k + 1:])
print(nums)   # không đổi
for k in range(len(nums)):
    print("xoá chỉ số", k, "->", nums[:k] + nums[k + 1:])

[10, 30] [10] [10, 30, 10]
[10, 30, 15, 10]
xoá chỉ số 0 -> [30, 15, 10]
xoá chỉ số 1 -> [10, 15, 10]
xoá chỉ số 2 -> [10, 30, 10]
xoá chỉ số 3 -> [10, 30, 15]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính điểm của một mảng — tính tay ví dụ 1 và 2

**`arr = [10, 30, 10]`:** `pre = [10, 10, 10]`, `suf = [10, 10, 10]`. Cắt `i = 0`: `pre[0] = 10 = suf[1]` ✔. Cắt `i = 1`: `pre[1] = 10 = suf[2]` ✔. Điểm 2.

**`arr = [2, 10, 14]`:** `pre = [2, 2, 2]`, `suf = [2, 2, 14]`. Cắt 0: `pre[0] = 2 = suf[1] = 2` ✔. Cắt 1: `pre[1] = 2 ≠ suf[2] = 14` ✘. Điểm 1.

### 4.2. Xoá phần tử thì chuyện gì xảy ra?

Ví dụ 1 gốc `[10, 30, 15, 10]`: không xoá thì `pre = [10, 10, 5, 5]`, `suf = [5, 5, 5, 10]`. Cắt 0: `10` vs `5` ✘; cắt 1: `10` vs `5` ✘; cắt 2: `5` vs `10` ✘ → điểm 0. Phần tử `15` "làm hỏng" GCD chung (kéo xuống 5). Xoá nó thì mọi GCD đều là 10.

Trực giác: **xoá một phần tử "kéo GCD xuống"** có thể làm cả hai phía bằng nhau. Xoá một phần tử "vô hại" (đã chia hết bởi GCD của mọi thứ quanh nó) không giúp ích gì mà còn làm giảm số vị trí cắt (mảng ngắn đi 1).

## 5. Cách 1 — Vét cạn: thử mọi cách xoá, dựng lại mảng và đếm

**Ý tưởng.** Có `n + 1` lựa chọn (không xoá, hoặc xoá chỉ số `k` cho `k = 0..n-1`). Với mỗi lựa chọn dựng `arr` rồi tính điểm bằng **định nghĩa** (GCD hai phía bằng nhau).

Để làm trọng tài thật "ngây thơ", hàm `score_def` tính GCD mỗi phía bằng `reduce` trên lát cắt: `O(m²)`. Hàm `score_prefsuf` dùng mảng tiền tố/hậu tố: `O(m)`.

In [7]:
def score_def(arr):
    """Điểm theo đúng định nghĩa đề bài: O(m^2)."""
    m = len(arr)
    return sum(reduce(gcd, arr[:i + 1]) == reduce(gcd, arr[i + 1:]) for i in range(m - 1))

def score_prefsuf(arr):
    """Điểm bằng mảng tiền tố / hậu tố: O(m)."""
    pre, suf = prefix_suffix(arr)
    return sum(pre[i] == suf[i + 1] for i in range(len(arr) - 1))

def max_valid_splits_bruteforce(nums, score=score_prefsuf):
    best = score(nums)                                  # không xoá
    for k in range(len(nums)):
        best = max(best, score(nums[:k] + nums[k + 1:]))   # xoá chỉ số k
    return best

for nums, expected in EXAMPLES:
    print(nums, max_valid_splits_bruteforce(nums), max_valid_splits_bruteforce(nums, score_def), expected)

[10, 30, 15, 10] 2 2 2
[2, 10, 14] 1 1 1
[2, 4] 0 0 0


**Độ phức tạp.** `n + 1` lựa chọn × `O(n)` gọi `gcd` mỗi lựa chọn = `O(n²)` lần `gcd`. Với `n = 10^5` là khoảng `10^10` — **quá chậm**.

**Nút thắt:** ta chấm điểm từng ứng viên xoá bằng `O(n)` và có `n` ứng viên. Cần (1) chấm điểm nhanh hơn / gọn hơn, và (2) giảm số ứng viên.

## 6. Bước đột phá

### 6.1. Công thức đóng cho điểm: `b - a`

Gọi `G = gcd(toàn mảng arr)`.

**Bổ đề 1.** Vị trí cắt `i` hợp lệ ⇔ `pre[i] = G` **và** `suf[i+1] = G`.

*Chứng minh.* Nếu `gcd(trái) = gcd(phải) = d` thì `G = gcd(trái, phải) = gcd(d, d) = d`. Ngược lại nếu hai phía cùng bằng `G` thì hiển nhiên bằng nhau. ∎

**Bổ đề 2.** `pre[i]` là bội của `G` và chia hết `pre[i-1]`. Nên khi `pre[i]` chạm `G` lần đầu (tại `a`), nó **giữ nguyên `G`** mãi (vì `G | pre[i] | pre[i-1] = G`). Tương tự `suf[i] = G` đúng với mọi `i <= b`, trong đó `b` là chỉ số **lớn nhất** có `suf[b] = G`.

**Hệ quả.** Các cắt hợp lệ là `i` với `a <= i` và `i + 1 <= b`, tức `a <= i <= b - 1`. Vậy

`điểm(arr) = max(0, b - a)`, với `a = chỉ số đầu tiên có pre = G`, `b = chỉ số cuối cùng có suf = G`.

Điều này cho phép chấm điểm bằng **hai vòng quét có thể dừng sớm**, không cần dựng hai mảng.

### 6.2. Chỉ cần thử xoá những chỉ số "làm đổi GCD"

Nói rằng GCD tiền tố **đổi tại `i`** nếu `pre[i] != pre[i-1]` (quy ước `i = 0` luôn tính là đổi). Tương tự GCD hậu tố đổi tại `i` nếu `suf[i] != suf[i+1]` (`i = n-1` luôn đổi).

**Bổ đề 3.** Nếu tại chỉ số `i` **cả** GCD tiền tố **và** hậu tố đều **không đổi**, thì xoá `nums[i]` **không làm tăng** điểm so với không xoá.

*Phác chứng minh.* Khi xoá `i`: GCD tiền tố của mọi phần tử sau `i` vẫn như cũ vì `pre[i-1] | nums[i]` nên chèn `nums[i]` hay không cũng không đổi `gcd`; tương tự hậu tố. Hai vị trí cắt cũ `(i-1 | i)` và `(i | i+1)` hợp thành một vị trí cắt mới `(i-1 | i+1)`; điều kiện của cả ba đều là `pre[i-1] == suf[i+1]`, nên ba vị trí này cùng đúng hoặc cùng sai. Sau khi gộp, số cắt hợp lệ **không tăng** (từ 2 xuống 1, hoặc 0 giữ 0). ∎

Mà theo 3.4 mỗi dãy chỉ đổi `<= log2(10^9) + 1 ≈ 31` lần, nên có tối đa `~62` ứng viên. Chấm điểm mỗi ứng viên `O(n)` ⇒ tổng `O(n log M)` lần `gcd`.

### 6.3. Quy tắc nếu… thì…

- Nếu `pre` đổi tại `i` **hoặc** `suf` đổi tại `i` → `i` là ứng viên xoá.
- Ngược lại → bỏ qua (không bao giờ tốt hơn "không xoá").
- Luôn thêm trường hợp **không xoá** vào so sánh.

In [8]:
def score_closed_form(arr):
    """Điểm = max(0, b - a) với a: chỉ số đầu có pre == G, b: chỉ số cuối có suf == G. O(m)."""
    G = reduce(gcd, arr, 0)
    a, g = 0, 0
    for i, x in enumerate(arr):
        g = gcd(g, x)
        if g == G:
            a = i
            break
    b, g = len(arr) - 1, 0
    for i in range(len(arr) - 1, -1, -1):
        g = gcd(g, arr[i])
        if g == G:
            b = i
            break
    return max(0, b - a)

for nums, _ in EXAMPLES:
    print(nums, score_closed_form(nums), score_def(nums))
rng = random.Random(37)
for _ in range(3000):
    arr = [rng.choice([1, 2, 3, 4, 6, 8, 12, 24]) for _ in range(rng.randint(1, 9))]
    assert score_closed_form(arr) == score_def(arr) == score_prefsuf(arr), arr
print("Công thức b - a khớp định nghĩa trên 3000 mảng ngẫu nhiên")

[10, 30, 15, 10] 0 0
[2, 10, 14] 1 1
[2, 4] 0 0
Công thức b - a khớp định nghĩa trên 3000 mảng ngẫu nhiên


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Vét cạn nhưng chấm điểm bằng công thức đóng

Giữ nguyên "thử mọi chỉ số", chỉ thay hàm chấm điểm bằng `score_closed_form` (quét có dừng sớm). Vẫn `O(n²)` trường hợp xấu nhưng hằng số tốt hơn, và là viên gạch cho cách cuối.

In [9]:
def max_valid_splits_all_removals(nums):
    best = score_closed_form(nums)
    for k in range(len(nums)):
        best = max(best, score_closed_form(nums[:k] + nums[k + 1:]))
    return best

for nums, expected in EXAMPLES:
    print(nums, max_valid_splits_all_removals(nums), expected)

[10, 30, 15, 10] 2 2
[2, 10, 14] 1 1
[2, 4] 0 0


### 7.2. Cách 3 — Chỉ thử xoá các chỉ số mà GCD tiền tố / hậu tố đổi

**Giải thích từng đoạn.**

1. `mark(arr)` trả về `pos[i] = True` nếu `gcd(g, arr[i]) != g` (với `g` là GCD các phần tử trước). Bắt đầu `g = 0` nên `pos[0] = True` (vì `gcd(0, x) = x ≠ 0`).
2. Gọi `mark` trên `nums` được các chỉ số GCD **tiền tố** đổi; gọi trên `nums[::-1]` rồi đảo lại được các chỉ số GCD **hậu tố** đổi.
3. Ứng viên = hợp của hai tập. Thử xoá từng ứng viên, chấm điểm, lấy max cùng điểm "không xoá".

In [10]:
def mark(arr):
    """pos[i] = True nếu gcd tiền tố đổi tại i (i = 0 luôn đổi)."""
    pos, g = [False] * len(arr), 0
    for i, x in enumerate(arr):
        ng = gcd(g, x)
        pos[i] = ng != g
        g = ng
    return pos

def candidates(nums):
    n = len(nums)
    fwd = mark(nums)
    bwd = mark(nums[::-1])               # bwd[j] ứng với chỉ số n - 1 - j của nums
    return [i for i in range(n) if fwd[i] or bwd[n - 1 - i]]

def max_valid_splits_candidates(nums):
    best = score_closed_form(nums)
    for k in candidates(nums):
        best = max(best, score_closed_form(nums[:k] + nums[k + 1:]))
    return best

print(candidates([10, 30, 15, 10]))      # chỉ số 0, 2, 3
for nums, expected in EXAMPLES:
    print(nums, candidates(nums), max_valid_splits_candidates(nums), expected)

[0, 2, 3]
[10, 30, 15, 10] [0, 2, 3] 2 2
[2, 10, 14] [0, 1, 2] 1 1
[2, 4] [0, 1] 0 0


**Độ phức tạp.** `candidates` tốn `O(n)` gọi `gcd`; có `<= ~62` ứng viên, mỗi ứng viên tốn `O(n)` (cắt lát + hai vòng quét) ⇒ `O(n log M)` lần `gcd`, mỗi `gcd` `O(log M)` ⇒ `O(n log² M)`. Bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Bảng cho `nums = [10, 30, 15, 10]`

| i | nums[i] | pre[i] | pre đổi? | suf[i] | suf đổi? | ứng viên? |
|---|---|---|---|---|---|---|
| 0 | 10 | 10 | có (đầu) | 5 | không (5 → 5) | **có** |
| 1 | 30 | 10 | không | 5 | không | không |
| 2 | 15 | 5 | có (10 → 5) | 5 | có (10 → 5) | **có** |
| 3 | 10 | 5 | không | 10 | có (cuối) | **có** |

Ứng viên `{0, 2, 3}`. Thử xoá từng ứng viên rồi chấm bằng công thức `b - a`: xoá `15` (chỉ số 2) ra điểm 2 là tốt nhất.

In [11]:
def trace_max_valid_splits(nums):
    n = len(nums)
    pre, suf = prefix_suffix(nums)
    fwd = mark(nums)
    bwd = mark(nums[::-1])[::-1]
    print("i  nums[i]  pre  suf  pre_đổi  suf_đổi")
    for i in range(n):
        print(f"{i:<2d} {nums[i]:<7d} {pre[i]:<4d} {suf[i]:<4d} {str(fwd[i]):<8s} {str(bwd[i])}")
    cands = [i for i in range(n) if fwd[i] or bwd[i]]
    print("ứng viên:", cands)
    best = score_closed_form(nums)
    print("không xoá -> điểm", best)
    for k in cands:
        arr = nums[:k] + nums[k + 1:]
        G = reduce(gcd, arr, 0)
        p, s = prefix_suffix(arr)
        a = next(i for i in range(len(arr)) if p[i] == G)
        b = max(i for i in range(len(arr)) if s[i] == G)
        sc = score_closed_form(arr)
        print(f"xoá chỉ số {k} -> arr={arr}, G={G}, a={a}, b={b}, điểm = max(0, b-a) = {sc}")
        best = max(best, sc)
    print("Đáp án =", best)
    return best

trace_max_valid_splits([10, 30, 15, 10])

i  nums[i]  pre  suf  pre_đổi  suf_đổi
0  10      10   5    True     False
1  30      10   5    False    False
2  15      5    5    True     True
3  10      5    10   False    True
ứng viên: [0, 2, 3]
không xoá -> điểm 0
xoá chỉ số 0 -> arr=[30, 15, 10], G=5, a=2, b=1, điểm = max(0, b-a) = 0
xoá chỉ số 2 -> arr=[10, 30, 10], G=10, a=0, b=2, điểm = max(0, b-a) = 2
xoá chỉ số 3 -> arr=[10, 30, 15], G=5, a=2, b=0, điểm = max(0, b-a) = 0
Đáp án = 2


2

### 8.2. Trace ví dụ 2 (`[2, 10, 14]`) — nơi "không xoá" thắng

Quan sát: mọi lần xoá đều đưa điểm xuống 0 (mảng chỉ còn 2 phần tử khác nhau), còn không xoá cho 1.

In [12]:
trace_max_valid_splits([2, 10, 14])

i  nums[i]  pre  suf  pre_đổi  suf_đổi
0  2       2    2    True     False
1  10      2    2    False    True
2  14      2    14   False    True
ứng viên: [0, 1, 2]
không xoá -> điểm 1
xoá chỉ số 0 -> arr=[10, 14], G=2, a=1, b=0, điểm = max(0, b-a) = 0
xoá chỉ số 1 -> arr=[2, 14], G=2, a=0, b=0, điểm = max(0, b-a) = 0
xoá chỉ số 2 -> arr=[2, 10], G=2, a=0, b=0, điểm = max(0, b-a) = 0
Đáp án = 1


1

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Để gọn, hai hàm trợ giúp `calc` (chấm điểm bằng `b - a`) và `mark` (đánh dấu chỗ GCD đổi) được viết bên trong phương thức.

In [13]:
class Solution:
    def maxValidSplits(self, nums: list[int]) -> int:
        n = len(nums)

        def calc(arr):
            # điểm = max(0, b - a): a = chỉ số đầu có gcd tiền tố == G, b = chỉ số cuối có gcd hậu tố == G
            G = 0
            for x in arr:
                G = gcd(G, x)
            a, g = 0, 0
            for i, x in enumerate(arr):
                g = gcd(g, x)
                if g == G:
                    a = i
                    break
            b, g = len(arr) - 1, 0
            for i in range(len(arr) - 1, -1, -1):
                g = gcd(g, arr[i])
                if g == G:
                    b = i
                    break
            return max(0, b - a)

        def mark(arr):
            pos, g = [False] * n, 0
            for i, x in enumerate(arr):
                ng = gcd(g, x)
                pos[i] = ng != g
                g = ng
            return pos

        fwd = mark(nums)
        bwd = mark(nums[::-1])

        ans = calc(nums)                           # không xoá
        for i in range(n):
            if fwd[i] or bwd[n - 1 - i]:           # chỉ thử xoá chỉ số làm đổi GCD tiền tố/hậu tố
                ans = max(ans, calc(nums[:i] + nums[i + 1:]))
        return ans


sol = Solution()
for nums, expected in EXAMPLES:
    assert sol.maxValidSplits(nums) == expected, nums
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Chấm điểm đúng.** Theo Bổ đề 1 và 2 (mục 6.1), điểm của `arr` đúng bằng `max(0, b - a)`. Hàm `calc` tìm `a` bằng cách quét từ trái và dừng ngay khi GCD tiền tố chạm `G`, tìm `b` quét từ phải tương tự. Nếu mảng có đúng một phần tử thì `a = b = 0` và điểm 0, khớp đề.

**(b) Không bỏ sót ứng viên tốt nhất.** Gọi `k` là chỉ số xoá tối ưu. Nếu tại `k` cả GCD tiền tố lẫn hậu tố **không đổi**, Bổ đề 3 nói điểm sau xoá `k` **không lớn hơn** điểm khi không xoá — mà ta luôn tính điểm "không xoá". Vậy tối ưu nằm trong tập {không xoá} ∪ {chỉ số mà tiền tố hoặc hậu tố đổi}.

**(c) Số ứng viên nhỏ.** Dãy GCD tiền tố là chuỗi ước giảm dần, mỗi lần đổi ít nhất giảm một nửa, nên đổi `<= log2(10^9) < 30` lần (cộng chỉ số đầu). Hậu tố tương tự. Tổng `<= ~62` ứng viên.

**(d) Ví dụ minh hoạ Bổ đề 3.** `[12, 6, 18, 6, 12]`: `pre = [12, 6, 6, 6, 6]`, `suf = [6, 6, 6, 6, 12]`. Chỉ số 2 (`18`) có `pre[2] = pre[1]` (không đổi) và `suf[2] = suf[3]` (không đổi). Điểm không xoá là 2; xoá chỉ số 2 ra `[12, 6, 6, 12]` có `G = 6`, `a = 1`, `b = 2` → điểm 1, thấp hơn — đúng với bổ đề.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `M = max(nums)` (≤ 10^9).

| Phần | Chi phí |
|---|---|
| `mark` hai lần | `O(n)` lần `gcd` |
| Số ứng viên | `O(log M)` (≤ ~62) |
| Chấm điểm một ứng viên | `O(n)` lần `gcd` (cắt lát + hai vòng quét) |
| **Tổng thời gian** | `O(n log M)` lần `gcd` = `O(n log² M)` |
| **Bộ nhớ** | `O(n)` cho `pos` và mảng cắt lát |

Một lần `gcd` trên số ≤ `M` tốn `O(log M)`, nhưng khi tính liên tiếp GCD của cả dãy thì tổng chi phí khấu hao còn nhỏ hơn. So sánh: vét cạn `O(n²)` lần `gcd`.

In [14]:
import time

rng = random.Random(4037)
big = [rng.randint(1, 10**9) * 2 for _ in range(100000)]     # gcd tiền tố thường chạm 2 rất nhanh
t = time.time(); r = Solution().maxValidSplits(big); print("n=1e5 ngẫu nhiên:", r, f"{time.time() - t:.2f}s")

# Trường hợp nhiều ứng viên: chuỗi chia đôi 2^29, 2^28, ..., lặp lại các phần tử bằng tích luỹ thừa
chain = [2**k for k in range(29, -1, -1)]
bigchain = chain + [2**29] * 99000 + chain[::-1]
t = time.time(); r = Solution().maxValidSplits(bigchain); print("n≈1e5 nhiều ứng viên:", r, f"{time.time() - t:.2f}s")
print("số ứng viên:", len(candidates(bigchain)))

n=1e5 ngẫu nhiên: 99995 0.06s


n≈1e5 nhiều ứng viên: 99001 0.20s


số ứng viên: 60


## 12. Bộ test

Ta kiểm: ví dụ đề, edge case, rồi đối chiếu tất cả cách với vét cạn theo **định nghĩa** (`score_def`) trên input ngẫu nhiên nhỏ có seed.

In [15]:
ALL = [
    max_valid_splits_bruteforce,
    max_valid_splits_all_removals,
    max_valid_splits_candidates,
    Solution().maxValidSplits,
]
EDGE = [
    ([5, 5], 1),                    # xoá một phần tử -> mảng độ dài 1 -> 0; không xoá -> 1
    ([5, 7], 0),
    ([7, 7, 7], 2),
    ([1, 1, 1, 1], 3),
    ([2, 4], 0),
    ([6, 10, 12, 6, 18], 2),        # xoá 10
    ([12, 6, 18, 6, 12], 2),
    ([8, 4, 16, 4, 8], 2),
    ([3, 5], 0),
    ([9, 3, 6, 3, 9], 2),
]
for fn in ALL:
    for nums, expected in EXAMPLES + EDGE:
        assert fn(nums) == expected, (fn.__name__, nums, fn(nums), expected)
print("Ví dụ đề + edge case: tất cả cách đúng")

rng = random.Random(403701)
for _ in range(1500):
    n = rng.randint(2, 9)
    pool = rng.choice([[1, 2, 3, 4, 6, 12], [2, 4, 8, 16], [6, 10, 15, 30], [5, 10, 20, 25, 50]])
    nums = [rng.choice(pool) for _ in range(n)]
    ref = max_valid_splits_bruteforce(nums, score_def)
    for fn in ALL:
        assert fn(nums) == ref, (fn.__name__, nums, fn(nums), ref)
print("1500 mảng ngẫu nhiên nhỏ: mọi cách khớp vét cạn theo định nghĩa")

Ví dụ đề + edge case: tất cả cách đúng
1500 mảng ngẫu nhiên nhỏ: mọi cách khớp vét cạn theo định nghĩa


In [16]:
# Mảng dài hơn, giá trị lớn: so Cách 3 với Cách 2 (cả hai dùng chấm điểm b - a, đã kiểm chứng)
rng = random.Random(7)
for _ in range(200):
    n = rng.randint(2, 60)
    g = rng.choice([1, 2, 6, 30])
    nums = [g * rng.choice([1, 2, 3, 5, 6, 10, 15, 30, 7, 14]) for _ in range(n)]
    assert Solution().maxValidSplits(nums) == max_valid_splits_all_removals(nums), nums
print("200 mảng dài 2..60: Cách 3 khớp Cách 2")

# Input không bị sửa
nums = [10, 30, 15, 10]
snapshot = nums[:]
Solution().maxValidSplits(nums)
assert nums == snapshot
print("nums không bị thay đổi")

200 mảng dài 2..60: Cách 3 khớp Cách 2
nums không bị thay đổi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bắt buộc phải xoá đúng một phần tử

**Code sai:** chỉ lấy max trên các lần xoá `k = 0..n-1`, quên trường hợp **không xoá**.
**Vì sao sai:** đề nói "tối đa một". Với `[2, 10, 14]` mọi lần xoá cho điểm 0 nhưng không xoá cho 1.
**Code đúng:** khởi tạo `ans = calc(nums)`.

In [17]:
def wrong_must_remove(nums):
    return max(score_closed_form(nums[:k] + nums[k + 1:]) for k in range(len(nums)))

print("sai:", wrong_must_remove([2, 10, 14]), " đúng:", Solution().maxValidSplits([2, 10, 14]))

sai: 0  đúng: 1


### Lỗi 2 — Dùng `0` làm "GCD của dãy rỗng" nhưng quên `gcd(0, x) = x` hoặc đặt `g = 1`

**Code sai:** khởi tạo `g = 1` rồi `g = gcd(g, x)`.
**Vì sao sai:** `gcd(1, x) = 1` với mọi `x` — GCD luôn bằng 1.
**Code đúng:** `g = 0` (phần tử trung tính của `gcd`).

In [18]:
g = 1
for x in [12, 18, 30]:
    g = gcd(g, x)
print("khởi tạo g = 1 ->", g, "(đúng phải là 6)")

khởi tạo g = 1 -> 1 (đúng phải là 6)


### Lỗi 3 — Đếm cả vị trí `i = m - 1` (phần phải rỗng)

**Code sai:** `sum(pre[i] == suf[i + 1] for i in range(m))` → `IndexError`; hoặc gán `suf[m] = 0` rồi so `pre[m-1] == 0` luôn sai nhưng dễ nhầm.
**Code đúng:** `range(m - 1)`.

### Lỗi 4 — Ánh xạ chỉ số sai khi lật mảng

**Code sai:** dùng `bwd[i]` thay cho `bwd[n - 1 - i]` khi `bwd = mark(nums[::-1])`.
**Vì sao sai:** `bwd[j]` ứng với chỉ số `n - 1 - j` của mảng gốc, nên tập ứng viên bị lệch. Lưu ý: bug kiểu này có thể **lọt qua nhiều test ngẫu nhiên** vì tập ứng viên thường dư; nhưng nó vẫn sai về nguyên tắc (có thể bỏ sót chỉ số cần xoá), nên hãy kiểm tra cả **tập ứng viên** chứ không chỉ đáp án cuối.
**Code đúng:** `bwd[n - 1 - i]`.

In [19]:
nums = [10, 30, 15, 10]
n = len(nums)
fwd, bwd = mark(nums), mark(nums[::-1])
dung = [i for i in range(n) if fwd[i] or bwd[n - 1 - i]]
sai = [i for i in range(n) if fwd[i] or bwd[i]]          # SAI: thiếu n - 1 - i
print("ứng viên đúng:", dung)
print("ứng viên sai: ", sai, "-> thiếu", sorted(set(dung) - set(sai)), "và thừa", sorted(set(sai) - set(dung)))

ứng viên đúng: [0, 2, 3]
ứng viên sai:  [0, 1, 2] -> thiếu [3] và thừa [1]


### Lỗi 5 — Chỉ thử xoá phần tử nhỏ nhất (hay lớn nhất)

**Vì sao sai:** thứ ta cần là phần tử **làm GCD đổi**, không liên quan độ lớn. Ví dụ `[6, 10, 12, 6, 18]`: phần tử cần xoá là `10`, không phải min (`6`) hay max (`18`).

In [20]:
nums = [6, 10, 12, 6, 18]
print("xoá min:", score_closed_form([x for i, x in enumerate(nums) if i != nums.index(min(nums))]))
print("xoá max:", score_closed_form([x for i, x in enumerate(nums) if i != nums.index(max(nums))]))
print("đáp án đúng:", Solution().maxValidSplits(nums))

xoá min: 0
xoá max: 0
đáp án đúng: 2


## 14. Mẹo & nhận dạng pattern

- **"GCD của đoạn / hai phía"** → mảng GCD tiền tố / hậu tố; nhớ `gcd(0, x) = x`.
- **"Dãy GCD (hoặc OR, AND) tiền tố đơn điệu"** → số lần đổi giá trị `O(log M)`; dùng để giảm số ứng viên hoặc số trạng thái.
- **"Thử xoá/đổi mỗi phần tử" với `n` lớn** → hỏi: phần tử nào thật sự thay đổi trạng thái? Thường chỉ `O(log)` phần tử.
- Phép "so sánh hai phía bằng nhau" của GCD ⇒ cả hai phía bằng GCD toàn mảng, thường cho công thức đóng.
- Luôn kiểm tra trường hợp "không làm gì" khi đề cho "tối đa".
- Khi đối chiếu, dùng hàm chấm điểm **theo định nghĩa** (`score_def`) làm chuẩn, không dùng công thức mình vừa suy ra.

## 15. Biến thể

1. **Phiên bản không xoá (điểm của một mảng):** chính là `max(0, b - a)`; viết hàm `count_valid_splits(nums)` trong `O(n)`.
2. **Xoá đúng một phần tử:** bỏ trường hợp không xoá khỏi phép max.
3. **Thay GCD bằng LCM / AND / OR / XOR:** AND và OR cũng đơn điệu (chỉ đổi `O(log M)` lần), XOR thì không.
4. **Xoá một đoạn liên tiếp:** cần prefix/suffix kết hợp, tương tự bài "Remove Subarray".
5. **Tổng thể:** cho `q` truy vấn xoá một chỉ số, trả điểm — dùng sparse table GCD để hợp phần tiền tố và hậu tố.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn, chấm theo định nghĩa | Thử mọi chỉ số xoá, dựng `pre/suf` | `O(n²)` lần `gcd` | `O(n)` | Trọng tài, `n ≤ 2000` |
| 2. Vét cạn + công thức `b - a` | Quét có dừng sớm | `O(n²)` (hằng số nhỏ) | `O(n)` | Học công thức đóng |
| 3. Ứng viên GCD đổi + `b - a` | Chỉ thử `O(log M)` chỉ số | `O(n log² M)` | `O(n)` | **Bản nộp**, `n = 10^5` |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `score_closed_form([6, 12, 18, 6])` bằng bao nhiêu? (Tìm `G`, `a`, `b`.)

**Bài 2 (tính tay).** `nums = [6, 10, 12, 6, 18]`. Lập bảng `pre`, `suf`, các ứng viên, rồi tìm đáp án của bài.

**Bài 3 (điền TODO).** Hoàn thiện `first_equal(arr, G)` trả về chỉ số đầu tiên mà GCD tiền tố bằng `G` (bài này là nửa đầu của `calc`).

**Bài 4 (tự viết lại).** Viết `my_max_valid_splits(nums)` theo Cách 3.

**Bài 5 (biến thể).** Viết `count_valid_splits(nums)` trả điểm của `nums` **khi không được xoá gì** trong một lần quét, không dùng mảng phụ.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def first_equal(arr, G):
    # TODO 1: g = 0
    # TODO 2: duyệt i, x bằng enumerate; g = gcd(g, x)
    # TODO 3: nếu g == G trả về i
    pass

# Bài 4 — TODO
def my_max_valid_splits(nums):
    # TODO: ans = điểm không xoá; tìm chỉ số GCD tiền tố/hậu tố đổi; thử xoá từng chỉ số đó
    pass

# Bài 5 — TODO
def count_valid_splits(nums):
    # TODO: G = gcd toàn mảng; a = first_equal(nums, G); b = chỉ số cuối có suf == G; return max(0, b - a)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tìm `G = gcd(6, 12, 18, 6)`.
- Mức 2: `G = 6`; `pre = [6, 6, 6, 6]`, nên `a = 0`; `suf = [6, 6, 6, 6]` nên `b = 3`.
- Mức 3: điểm `max(0, 3 - 0) = 3`.

**Bài 2.**
- Mức 1: `pre = [6, 2, 2, 2, 2]`, `suf = [2, 2, 6, 6, 18]`.
- Mức 2: `pre` đổi tại 0, 1; `suf` đổi tại 4 (18), 3 (6), 2 (6 → gcd(12, 6) = 6 không đổi), 1 (gcd(10, 6) = 2 đổi).
- Mức 3: thử xoá chỉ số 1 (`10`) được `[6, 12, 6, 18]` với `G = 6`, `a = 0`, `b = 2` → điểm 2. Đáp án **2** (không xoá cho điểm 0).

**Bài 3.**
- Mức 1: giống vòng quét trong `score_closed_form`.
- Mức 2: dừng ngay khi `g == G`.
- Mức 3: `return i` bên trong vòng `for`.

**Bài 4.**
- Mức 1: dựng `fwd`, `bwd` bằng `mark`.
- Mức 2: nhớ `bwd[n - 1 - i]`.
- Mức 3: `ans = max(ans, calc(nums[:i] + nums[i + 1:]))` cho từng ứng viên.

**Bài 5.**
- Mức 1: dùng công thức `b - a`.
- Mức 2: `b` quét từ phải.
- Mức 3: không cần mảng `pre`/`suf`, chỉ biến `g`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1, 2
assert score_closed_form([6, 12, 18, 6]) == 3
assert Solution().maxValidSplits([6, 10, 12, 6, 18]) == 2
assert score_closed_form([6, 10, 12, 6, 18]) == 0
assert score_closed_form([6, 12, 6, 18]) == 2

# Bài 3
def first_equal_ans(arr, G):
    g = 0
    for i, x in enumerate(arr):
        g = gcd(g, x)
        if g == G:
            return i
    return -1

assert first_equal_ans([6, 10, 12, 6, 18], 2) == 1
assert first_equal_ans([6, 12, 6, 18], 6) == 0
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_max_valid_splits_ans(nums):
    n = len(nums)
    fwd, bwd = mark(nums), mark(nums[::-1])
    ans = score_closed_form(nums)
    for i in range(n):
        if fwd[i] or bwd[n - 1 - i]:
            ans = max(ans, score_closed_form(nums[:i] + nums[i + 1:]))
    return ans

# Bài 5
def count_valid_splits_ans(nums):
    G = reduce(gcd, nums, 0)
    a = first_equal_ans(nums, G)
    g, b = 0, len(nums) - 1
    for i in range(len(nums) - 1, -1, -1):
        g = gcd(g, nums[i])
        if g == G:
            b = i
            break
    return max(0, b - a)

rng = random.Random(99)
for _ in range(500):
    nums = [rng.choice([1, 2, 3, 4, 6, 12]) for _ in range(rng.randint(1, 9))]
    assert count_valid_splits_ans(nums) == score_def(nums)
    if len(nums) >= 2:
        assert my_max_valid_splits_ans(nums) == Solution().maxValidSplits(nums)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
def calc(arr):                       # điểm = max(0, b - a)
    G = gcd của cả arr
    a = chỉ số đầu tiên có gcd tiền tố == G
    b = chỉ số cuối cùng có gcd hậu tố == G
    return max(0, b - a)

def mark(arr):                       # chỗ gcd tiền tố đổi
    g = 0
    for i, x in enumerate(arr):
        ng = gcd(g, x); pos[i] = ng != g; g = ng

ans = calc(nums)                                  # không xoá
for i in range(n):
    if fwd[i] or bwd[n - 1 - i]:                  # chỉ ứng viên
        ans = max(ans, calc(nums[:i] + nums[i+1:]))
```

**Checklist trước khi submit:**

- [ ] Tính cả trường hợp **không xoá**.
- [ ] `g` khởi tạo bằng `0`, không phải `1`.
- [ ] Chỉ vị trí cắt `0 <= i < m - 1`.
- [ ] Ánh xạ `bwd[n - 1 - i]` khi lật mảng.
- [ ] Ứng viên = tiền tố **hoặc** hậu tố đổi (kể cả chỉ số 0 và n-1).
- [ ] Mảng độ dài 1 sau khi xoá → điểm 0.
- [ ] Đối chiếu bằng test ngẫu nhiên với `score_def` theo định nghĩa.